In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez']


# DB-Specific

In [3]:
from lib import mymixtapez
mio   = mymixtapez.MusicDBIO(verbose=False)
webio = mymixtapez.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant MyMixTapez DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/MyMixTapez


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:   {0}".format(len(localArtists.get())))
print("   Master Artists:  {0}".format(len(masterArtists.get())))
print("   Errors:          {0}".format(len(errors.get())))
print("   Search Artists:  {0}".format(searchArtists.shape[0]))
print("   Known IDs:       {0}".format(len(knownArtists)))

MyMixTapez Search Results
   Local Artists:   1645
   Master Artists:  0
   Errors:          10
   Search Artists:  1650
   Known IDs:       1645


In [ ]:
mdbio = mymixtapez.MusicDBIO(verbose=True)
mdbio.prd.parse(0, force=True, verbose=True)
#ifile='/Volumes/Piggy/Discog/artists-mymixtapez/11/18311.p'
#mdbio.prd.rawio.getArtistData(ifile).show()

# Download Artist Data

In [ ]:
mio   = mymixtapez.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = mymixtapez.RawWebData(debug=False)

In [ ]:
useSearchData  = True
useStarterData = False

if useStarterData is True:
    artistNames      = io.get("starter.p")
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())]["Name"]

print("{0} Search Results".format(db))
print("   Available Names:      {0}".format(len(artistNames)))
print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,artistName) in enumerate(artistNamesToGet.iteritems()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(5.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
from lib.mymixtapez import moveLocalFiles
moveLocalFiles()

In [ ]:
mdbio = mymixtapez.MusicDBIO(local=False)

In [ ]:
mdbio.prd.parse(0, verbose=True, force=True)

In [6]:
from utils import PoolIO
pio = PoolIO("MyMixTapez")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

poolIO(numProcs=3)
  ==> ParseFunction: parse
  ==> ModVals:       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Process [Running imap multiprocessing for 100 mod values ...] Start    ==> Time Is 2022-05-03 14:27:05


100%|█████████████████████████████████████████| 100/100 [00:02<00:00, 45.22it/s]


Process [Running imap multiprocessing for 100 mod values ...] Ran For 2 Seconds    ==> Time Is 2022-05-03 14:27:07
poolMetaIO(numProcs=3)
  ==> MakeFunction: make
  ==> ModVals:      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Process [Running imap multiprocessing for 100 mod values ...] Start    ==> Time Is 2022-05-03 14:27:07


100%|█████████████████████████████████████████| 100/100 [00:02<00:00, 40.57it/s]


Process [Running imap multiprocessing for 100 mod values ...] Ran For 3 Seconds    ==> Time Is 2022-05-03 14:27:10
  ====> Saving [1645] ID => Name Basic Summary Data
  ====> Saving [1645] ID => Ref Basic Summary Data
  ====> Saving [1645] ID => Num Albums Basic Summary Data
  ====> Saving [1645] ID => Media Counts Counts Summary Data
  ====> Saving [1645] ID => Album Media Summary Data
  ====> Saving [1645] ID => SingleEP Media Summary Data
  ====> Not Saving ID => Appearance Media Summary Data
  ====> Not Saving ID => Technical Media Summary Data
  ====> Not Saving ID => Mix Media Summary Data
  ====> Not Saving ID => Bootleg Media Summary Data
  ====> Not Saving ID => AltMedia Media Summary Data
  ====> Not Saving ID => Other Media Summary Data


# Create Starter

In [ ]:
from ioutils import HTMLIO, FileIO
import json
hio = HTMLIO()
io = FileIO()
artists = {}
bsdata = hio.get(io.get("../../sandbox/recent_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["album-recents-page"]['success']['recents']:
    artists.update({artist['id']: artist['name'] for artist in item['artists']['main']})
    
bsdata = hio.get(io.get("../../sandbox/trending_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["trending-artists-page"]['success']['items']:
    artists.update({item['id']: item['name']})

In [ ]:
io.save(idata=Series(artists), ifile="starter.p")

In [ ]:
from collections import Counter
import json
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
mio = mymixtapez.MusicDBIO(local=False)
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p", debug=False):
        bsdata = hio.get(io.get(ifile))
        jTag = bsdata.find("script", {"type": "application/json"})
        jData = json.loads(jTag.text.replace('&q;', '"'))
        for item in jData['artist-page']['success']['alsoAppears']:
            for artist in item['artists']['main']:
                refs.append(artist)
    if (modVal+1) % 25 == 0:
        print(modVal+1,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df['id'].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df['id']
df.index.name = None
df = df.drop(['id'], axis=1)
df.columns = ["Name"]
df = df.join(N)
df = df.sort_values(by='Num', ascending=False)
print(df.shape)
mio = mymixtapez.MusicDBIO(local=False)
mio.data.saveSearchArtistData(data=df)